# Remoção Espectral de Fingerprint — BARS & MEAN nos nossos dados

Aplica os dois ataques de `spectral_fingerprint_removal.py` (Wesselkamp et al., DLS 2022) ao nosso problema:

- **BARS** (untargeted): zera uma borda de altas frequências do espectro — onde mora a grade de upsampling. 1 parâmetro: `width`.
- **MEAN** (targeted): aprende a digital = média(espectro StyleGAN fake) − média(espectro real) e subtrai dela. 1 parâmetro: `factor`.

O que este notebook faz:
1. **Ajusta e visualiza a digital média do StyleGAN** (140k) — *ver* o atalho que a ResNet aprende.
2. Aplica os dois ataques em imagens de exemplo (antes/depois + espectro + PSNR/SSIM).
3. **Diagnóstico:** quanto cada ataque, em cada intensidade, derruba a separabilidade espectral real-vs-fake — e a que custo de qualidade. Acha o ponto ótimo (`width`, `factor`).
4. **BARS cross-generator:** como o ataque untargeted afeta cada um dos 8 geradores do ArtiFact.

Saídas: `artifacts/mean_fingerprint/` (a digital `.npy`) e figuras em `reports/figures/sfr_*.png`.

> Tudo CPU/numpy, sem treinar CNN — minutos. A digital ajustada aqui serve depois como augmentation de treino.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import gaussian_filter
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

# o modulo de ataques esta nesta mesma pasta
sys.path.insert(0, str(Path.cwd()))
import spectral_fingerprint_removal as sfr
# split dev/test do ArtiFact
sys.path.insert(0, str(Path.cwd().parent / "notebooks_140k"))
from aug_utils import artifact_split

PROJECT_ROOT = Path.cwd().resolve().parent
_envf = PROJECT_ROOT / "data_root.env"
DATA_ROOT = Path(_envf.read_text().strip()) if _envf.exists() else PROJECT_ROOT / "data"
RAW_140K     = DATA_ROOT / "raw" / "140k_faces" / "real_vs_fake" / "real-vs-fake"
ARTIFACT_DIR = DATA_ROOT / "raw" / "artifact_faces"
FP_DIR  = PROJECT_ROOT / "artifacts" / "mean_fingerprint"
FIGS_DIR = PROJECT_ROOT / "reports" / "figures"
FP_DIR.mkdir(parents=True, exist_ok=True); FIGS_DIR.mkdir(parents=True, exist_ok=True)

SIZE     = 256
N_FIT    = 1500   # imagens p/ ajustar a digital media
N_DIAG   = 250    # imagens por classe no diagnostico
SEED     = 42
rng = np.random.default_rng(SEED)
print("140k:", RAW_140K.exists(), "| ArtiFact:", ARTIFACT_DIR.exists())

## 1. A digital média do StyleGAN

`fingerprint = média(espectro StyleGAN-fake) − média(espectro CelebA-real)`, ajustada nas imagens de treino do 140k (o gerador e os reais que a ResNet realmente vê). Salva em disco e reutiliza.

In [ ]:
FP_PATH = FP_DIR / f"stylegan_140k_{SIZE}.npy"
if FP_PATH.exists():
    fingerprint = np.load(FP_PATH)
    print("digital carregada de", FP_PATH.name)
else:
    print(f"ajustando a digital (size={SIZE}, ate {N_FIT} imgs/classe)...")
    fingerprint = sfr.fit_mean_fingerprint(RAW_140K / "train" / "fake",
                                           RAW_140K / "train" / "real",
                                           size=SIZE, max_images=N_FIT)
    np.save(FP_PATH, fingerprint)
    print("salva em", FP_PATH)

mag = np.abs(fingerprint)
print("shape:", fingerprint.shape, "| magnitude media/max:", f"{mag.mean():.4f} / {mag.max():.4f}")

In [ ]:
# visualiza a digital (log-magnitude, media dos canais) — DC mascarado para nao ofuscar
logmag = np.log1p(np.abs(fingerprint).mean(axis=2))
c = SIZE // 2
view = logmag.copy(); view[c-2:c+3, c-2:c+3] = np.nan   # mascara o DC (centro)

fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
im0 = ax[0].imshow(view, cmap="magma")
ax[0].set_title("Digital MEAN do StyleGAN (log-magnitude)\npicos fora do centro = artefatos periodicos")
ax[0].axis("off"); plt.colorbar(im0, ax=ax[0], fraction=0.046)

# perfil radial da magnitude da digital
yy, xx = np.indices((SIZE, SIZE)); r = np.hypot(xx - c, yy - c).astype(int)
rad = np.bincount(r.ravel(), np.abs(fingerprint).mean(2).ravel()) / np.maximum(np.bincount(r.ravel()), 1)
ax[1].plot(rad[:SIZE // 2], lw=2)
ax[1].set_title("Perfil radial da digital"); ax[1].set_xlabel("frequencia radial")
ax[1].set_ylabel("|digital| media"); ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS_DIR / "sfr_digital_stylegan.png", dpi=130, bbox_inches="tight")
plt.show()

## 2. Os dois ataques numa imagem

Pega fakes do StyleGAN e aplica BARS e MEAN. Mostra original | BARS | MEAN, os espectros, e PSNR/SSIM (quanto a imagem mudou).

In [ ]:
def ssim_gray(a01, b01, sigma=1.5):
    a = a01.mean(2) if a01.ndim == 3 else a01
    b = b01.mean(2) if b01.ndim == 3 else b01
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    ma, mb = gaussian_filter(a, sigma), gaussian_filter(b, sigma)
    va = gaussian_filter(a * a, sigma) - ma ** 2
    vb = gaussian_filter(b * b, sigma) - mb ** 2
    cov = gaussian_filter(a * b, sigma) - ma * mb
    s = ((2 * ma * mb + C1) * (2 * cov + C2)) / ((ma ** 2 + mb ** 2 + C1) * (va + vb + C2))
    return float(s.mean())

def logspec(img01):
    g = img01.mean(2) if img01.ndim == 3 else img01
    return np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(g))))

sty_files = sorted((RAW_140K / "train" / "fake").glob("*.jpg"))
sample = [sty_files[i] for i in rng.choice(len(sty_files), 3, replace=False)]
BARS_W, MEAN_F = 24, 1.0

fig, axes = plt.subplots(3, 6, figsize=(17, 9))
for row, p in enumerate(sample):
    orig = sfr.load_image(p, size=SIZE)
    b = np.clip(sfr.attack_bars(orig, width=BARS_W), 0, 1)
    m = np.clip(sfr.attack_mean(orig, fingerprint, factor=MEAN_F), 0, 1)
    panels = [(orig, "original"),
              (b, f"BARS w={BARS_W}\nPSNR {sfr.psnr(orig, b):.1f} SSIM {ssim_gray(orig, b):.2f}"),
              (m, f"MEAN f={MEAN_F}\nPSNR {sfr.psnr(orig, m):.1f} SSIM {ssim_gray(orig, m):.2f}")]
    for col, (im, ttl) in enumerate(panels):
        axes[row, col].imshow(im); axes[row, col].set_title(ttl, fontsize=8); axes[row, col].axis("off")
        axes[row, col + 3].imshow(logspec(im), cmap="magma"); axes[row, col + 3].axis("off")
        if row == 0:
            axes[row, col + 3].set_title(["espectro orig", "espectro BARS", "espectro MEAN"][col], fontsize=8)
plt.suptitle("StyleGAN fake: imagem (esq) e espectro (dir) — original vs BARS vs MEAN")
plt.tight_layout()
plt.savefig(FIGS_DIR / "sfr_antes_depois.png", dpi=120, bbox_inches="tight")
plt.show()

## 3. Diagnóstico — destruição do atalho vs custo de qualidade

Para cada intensidade: aplica o ataque **só nos fakes** (cenário do atacante), mede a AUC de um probe espectral real-vs-fake (queda = atalho removido) e o PSNR/SSIM médio (custo). O ponto ótimo derruba a AUC perto de 0.5 mantendo a qualidade alta.

In [ ]:
def radial_feat(img01):
    g = img01.mean(2) if img01.ndim == 3 else img01
    F = np.fft.fftshift(np.abs(np.fft.fft2(g))) ** 2
    rr = np.hypot(*[a - SIZE // 2 for a in np.indices((SIZE, SIZE))][::-1]).astype(int)
    prof = np.bincount(rr.ravel(), F.ravel()) / np.maximum(np.bincount(rr.ravel()), 1)
    return np.log1p(prof[:SIZE // 2])

# amostras fixas: StyleGAN fakes + CelebA reais
real_files = sorted((RAW_140K / "train" / "real").glob("*.jpg"))
fk = [sfr.load_image(sty_files[i], size=SIZE) for i in rng.choice(len(sty_files), N_DIAG, replace=False)]
rl = [sfr.load_image(real_files[i], size=SIZE) for i in rng.choice(len(real_files), N_DIAG, replace=False)]
Xr = np.stack([radial_feat(im) for im in rl])

def probe_auc(fake_imgs):
    Xf = np.stack([radial_feat(im) for im in fake_imgs])
    X = np.concatenate([Xr, Xf]); y = np.r_[np.zeros(len(Xr)), np.ones(len(Xf))]
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    return cross_val_score(clf, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring="roc_auc").mean()

auc0 = probe_auc(fk)
rows = []
for w in [4, 8, 12, 16, 24, 32, 48, 64]:
    att = [np.clip(sfr.attack_bars(im, width=w), 0, 1) for im in fk]
    rows.append(("bars", w, probe_auc(att),
                 np.mean([sfr.psnr(o, a) for o, a in zip(fk, att)]),
                 np.mean([ssim_gray(o, a) for o, a in zip(fk, att)])))
for f in [0.25, 0.5, 0.75, 1.0, 1.5, 2.0]:
    att = [np.clip(sfr.attack_mean(im, fingerprint, factor=f), 0, 1) for im in fk]
    rows.append(("mean", f, probe_auc(att),
                 np.mean([sfr.psnr(o, a) for o, a in zip(fk, att)]),
                 np.mean([ssim_gray(o, a) for o, a in zip(fk, att)])))
import pandas as pd
diag = pd.DataFrame(rows, columns=["ataque", "valor", "auc", "psnr", "ssim"])
print(f"AUC espectral real-vs-fake (StyleGAN), imagens limpas: {auc0:.3f}\n")
print(diag.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for atk, mk in [("bars", "o"), ("mean", "s")]:
    d = diag[diag.ataque == atk]
    axes[0].plot(d.valor, d.auc, marker=mk, label=atk)
    axes[1].plot(d.ssim, d.auc, marker=mk, label=atk)
axes[0].axhline(auc0, color="gray", ls="--", lw=1, label=f"limpa ({auc0:.2f})")
axes[0].axhline(0.5, color="red", ls=":", lw=1, label="chance")
axes[0].set_xlabel("intensidade (width / factor)"); axes[0].set_ylabel("AUC espectral real-vs-fake")
axes[0].set_title("Destruicao do atalho vs intensidade"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].axhline(0.5, color="red", ls=":", lw=1)
axes[1].set_xlabel("SSIM (qualidade preservada)"); axes[1].set_ylabel("AUC espectral real-vs-fake")
axes[1].set_title("Trade-off: canto inferior direito = ideal\n(atalho destruido, imagem intacta)")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3); axes[1].invert_xaxis()
plt.tight_layout()
plt.savefig(FIGS_DIR / "sfr_diagnostico.png", dpi=130, bbox_inches="tight")
plt.show()

# ponto otimo: menor AUC com SSIM >= 0.85
ok = diag[diag.ssim >= 0.85]
if len(ok):
    best = ok.loc[ok.auc.idxmin()]
    print(f"Sugestao (SSIM>=0.85): {best.ataque} = {best.valor}  -> AUC {best.auc:.3f}, SSIM {best.ssim:.2f}, PSNR {best.psnr:.1f}")

## 4. BARS cross-generator (untargeted)

O BARS não precisa de digital ajustada — funciona em qualquer gerador. Aplica-o a cada gerador do ArtiFact (metade `dev`) e mede a queda da AUC espectral, para ver onde o ataque untargeted ajuda.

In [ ]:
def list_by_source(folder):
    g = {}
    for p in folder.glob("*.*"):
        s = p.name.split("__")[0] if "__" in p.name else "?"
        g.setdefault(s, []).append(p)
    return {k: sorted(v) for k, v in g.items()}

fake_groups = list_by_source(ARTIFACT_DIR / "fake")
real_groups = list_by_source(ARTIFACT_DIR / "real")
GEN = sorted(fake_groups)

art_real = []
for s in sorted(real_groups):
    dev = artifact_split(real_groups[s], which="dev")
    art_real += [sfr.load_image(dev[i], size=SIZE) for i in rng.choice(len(dev), 30, replace=False)]
Xar = np.stack([radial_feat(im) for im in art_real])

def auc_gen(fimgs):
    Xf = np.stack([radial_feat(im) for im in fimgs])
    X = np.concatenate([Xar, Xf]); y = np.r_[np.zeros(len(Xar)), np.ones(len(Xf))]
    n = min(len(Xar), len(Xf))
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    return cross_val_score(clf, X, y, cv=StratifiedKFold(4, shuffle=True, random_state=SEED), scoring="roc_auc").mean()

W = 24
recs = []
for g in GEN:
    dev = artifact_split(fake_groups[g], which="dev")
    imgs = [sfr.load_image(dev[i], size=SIZE) for i in rng.choice(len(dev), 80, replace=False)]
    a0 = auc_gen(imgs)
    aB = auc_gen([np.clip(sfr.attack_bars(im, width=W), 0, 1) for im in imgs])
    recs.append({"gerador": g, "auc_limpa": round(a0, 3), f"auc_bars_w{W}": round(aB, 3), "delta": round(aB - a0, 3)})
cg = pd.DataFrame(recs).sort_values("delta")
print(f"BARS (width={W}) por gerador — queda da AUC espectral:\n")
print(cg.to_string(index=False))

## 5. Leitura e próximos passos

- A **Seção 1** mostra a digital do StyleGAN: se houver picos fora do centro, é o artefato periódico de upsampling — o atalho.
- A **Seção 3** dá os valores: o ponto da curva que derruba a AUC perto de 0.5 mantendo SSIM alto é a intensidade certa de cada ataque. O painel de trade-off (canto inferior direito) é o que importa.
- A **Seção 4** mostra onde o BARS (untargeted) ajuda cross-generator — provavelmente forte nos GANs com grade (cips, projected_gan), fraco na difusão.

**Onde cada ataque vai:**
- **BARS** (`width` ótimo da Seção 3) → entra no pool de augmentation (já está no `aug_utils`) e na busca adversarial.
- **MEAN** (`factor` ótimo) → augmentation de treino *targeted* no StyleGAN: a digital salva (`stylegan_140k_256.npy`) é removida dos fakes do treino, atacando exatamente o atalho da ResNet.

**Próximo passo:** plugar BARS e/ou MEAN no treino (notebook 02) com os valores achados aqui e medir o cross-generator na metade `test`.